# WorldGap demo notebook

Runs entirely on synthetic data — no Kaggle download, MediaPipe extraction, or
GPU access required (see `ROADMAP.md` Phase 0/1 for what's still blocked on
those). This notebook demonstrates the three things the technical spec treats
as load-bearing:

1. **V1 (perception) gap quantification** — Frechet distance + MMD between a
   clean and a synthetically tremor-perturbed rollout set, including a
   sanity check that the gap score actually scales with perturbation severity.
2. **Reusability across modalities** — the identical `GapAnalyzer` class,
   unmodified, computing a **V2 (actuation)** gap by only changing
   `config.modality`.
3. **The validation harness's anti-cherry-picking enforcement** — a
   pre-registered condition set correlated against a (proxy) ground truth via
   Spearman + bootstrap CI, with the rejection check demonstrated directly.

Configs below are deliberately small (few epochs, small hidden dims) so this
runs in well under a minute — this is a wiring and sanity-check
demonstration, not a convergence benchmark. Real V1 results need real HaGRID
+ MediaPipe data; real V2 results need the digitized Ogawa et al. curve or
real ForceHand glove logs — both are genuine Phase 0/1/6 blockers this
notebook does not attempt to work around.

In [1]:
import numpy as np

from worldgap import GapAnalyzer, GapConfig, ReportEntry, Rollout, generate_report
from worldgap.config import EncoderConfig, TrainingConfig, WorldModelConfig
from worldgap.data.loaders.synthetic_perturb import inject_tremor
from worldgap.data.rollout import PERCEPTION_STATE_DIM
from worldgap.validation.harness import ConditionResult, ValidationHarness

RNG_SEED = 0
print("worldgap imported successfully.")

worldgap imported successfully.


## Part 1 — V1: perception domain gap (clean vs. tremor)

In [2]:
def make_clean_perception_rollout(t=40, seed=0):
    rng = np.random.default_rng(seed)
    return Rollout(
        modality="perception",
        source="synthetic",
        condition={"lighting": "clean"},
        frame_rate_hz=30.0,
        states=rng.normal(scale=0.05, size=(t, PERCEPTION_STATE_DIM)),
        presence_mask=np.ones((t, PERCEPTION_STATE_DIM)),
        timestamps_ms=np.arange(t) * (1000.0 / 30.0),
    )


clean_rollouts = [make_clean_perception_rollout(seed=i) for i in range(15)]
tremor_rollouts = [
    inject_tremor(r, frequency_hz=5.0, amplitude=0.4, seed=i) for i, r in enumerate(clean_rollouts)
]
print(f"Built {len(clean_rollouts)} clean and {len(tremor_rollouts)} tremor-perturbed perception rollouts.")

Built 15 clean and 15 tremor-perturbed perception rollouts.


In [3]:
perception_config = GapConfig(
    modality="perception",
    state_dim=PERCEPTION_STATE_DIM,
    encoder=EncoderConfig(d_model=32, n_layers=1, n_heads=2, dim_feedforward=64),
    world_model=WorldModelConfig(context_frames=6, predict_frames=3, summary_dim=8),
    training=TrainingConfig(max_epochs=8, batch_size=4, seed=RNG_SEED),
)
perception_analyzer = GapAnalyzer(perception_config)
fit_report = perception_analyzer.fit(clean_rollouts)
print(fit_report)

{'final_loss': 0.31562578678131104, 'n_steps': 32, 'n_skipped_rollouts': 0, 'collapsed': False}


In [4]:
perception_result = perception_analyzer.compute_gap(clean_rollouts, tremor_rollouts)
print(f"Frechet distance: {perception_result.frechet.distance:.6f}")
print(f"MMD^2:            {perception_result.mmd.mmd_squared:.6f}")
print(f"Confidence:       {perception_result.confidence}")
if perception_result.warnings:
    print("Warnings:", perception_result.warnings)

Frechet distance: 0.000977
MMD^2:            0.317838
Confidence:       low
Warnings: ['low sample-size confidence (n_source=15, n_target=15, latent_dim=8) — see spec Section 7.3']


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(


### Sanity check: does the gap score scale with perturbation severity?

A domain-gap metric that reports the same score regardless of how severe the
perturbation is would be useless. This sweeps tremor amplitude and checks
that both metrics move in the expected direction — this is the actual claim
the tool is making, not just that it returns a number.

In [5]:
print(f"{'amplitude':>10} | {'frechet':>12} | {'mmd^2':>10}")
for amp in [0.05, 0.15, 0.4, 0.8]:
    perturbed = [inject_tremor(r, frequency_hz=5.0, amplitude=amp, seed=i) for i, r in enumerate(clean_rollouts)]
    r = perception_analyzer.compute_gap(clean_rollouts, perturbed)
    print(f"{amp:>10} | {r.frechet.distance:>12.6f} | {r.mmd.mmd_squared:>10.6f}")

 amplitude |      frechet |      mmd^2
      0.05 |     0.000001 |  -0.053322
      0.15 |     0.000037 |  -0.008243
       0.4 |     0.000977 |   0.317838
       0.8 |     0.009438 |   0.645713


### Generating a report (spec 9.3)

Bundles the `GapResult`, a condition table, and a Frechet/MMD trend plot into
a single self-contained HTML file.

In [6]:
report_path = generate_report(
    [ReportEntry("clean_vs_tremor", perception_result)],
    "demo_report.html",
    title="WorldGap Demo -- Perception Gap (clean vs. tremor)",
)
print(f"Report written to: {report_path.resolve()}")

Report written to: /home/claude/worldgap/worldgap/notebooks/demo_report.html


## Part 2 — V2: actuation domain gap, same `GapAnalyzer` class

This is the concrete test of the reusability claim (spec Section 3 / 9.1):
everything below reuses the identical `GapAnalyzer` class from Part 1 --
only `config.modality`/`config.state_dim` change, and a different encoder
gets selected internally. No new analysis class, no copy-pasted pipeline.

The actuation data below is a toy two-branch hysteresis signal (commanded
pressure -> resulting joint angle), standing in for the real PGM
pressure-response data spec 5.5/12.13 describes -- see `ROADMAP.md` Phase 6
for what's still blocked (digitizing the real Ogawa et al. curve).

In [7]:
def make_actuation_rollout(t=60, seed=0, hysteresis_strength=1.0):
    rng = np.random.default_rng(seed)
    pressure = np.abs(np.sin(np.linspace(0, 2 * np.pi, t))) * rng.uniform(0.8, 1.2)
    d_pressure = np.gradient(pressure)
    # toy hysteresis: response lags pressure depending on load/unload direction
    response = pressure - hysteresis_strength * 0.1 * np.sign(d_pressure) * pressure
    states = np.stack([pressure, response], axis=1)
    return Rollout(
        modality="actuation",
        source="synthetic",
        condition={"unit": f"pgm_{seed}"},
        frame_rate_hz=100.0,
        states=states,
        presence_mask=np.ones_like(states),
        timestamps_ms=np.arange(t) * 10.0,
    )


sim_rollouts = [make_actuation_rollout(seed=i, hysteresis_strength=1.0) for i in range(15)]
real_proxy_rollouts = [make_actuation_rollout(seed=i + 100, hysteresis_strength=10.0) for i in range(15)]
print(f"Built {len(sim_rollouts)} sim and {len(real_proxy_rollouts)} 'real-proxy' actuation rollouts.")

Built 15 sim and 15 'real-proxy' actuation rollouts.


In [8]:
actuation_config = GapConfig(
    modality="actuation",
    state_dim=2,
    encoder=EncoderConfig(d_model=16, n_layers=1, n_heads=2, dim_feedforward=32),
    world_model=WorldModelConfig(context_frames=6, predict_frames=3, summary_dim=4),
    training=TrainingConfig(max_epochs=8, batch_size=4, seed=RNG_SEED),
)
actuation_analyzer = GapAnalyzer(actuation_config)  # same class as perception_analyzer
actuation_analyzer.fit(sim_rollouts)
actuation_result = actuation_analyzer.compute_gap(sim_rollouts, real_proxy_rollouts)

print(f"Frechet distance: {actuation_result.frechet.distance:.6f}")
print(f"MMD^2:            {actuation_result.mmd.mmd_squared:.6f}")
print(f"Confidence:       {actuation_result.confidence}")

assert type(perception_analyzer) is type(actuation_analyzer) is GapAnalyzer
print("\nConfirmed: perception and actuation gaps were computed by the identical GapAnalyzer class.")

Frechet distance: 0.000155
MMD^2:            0.314333
Confidence:       low

Confirmed: perception and actuation gaps were computed by the identical GapAnalyzer class.


**On the numbers above**: MMD² is the more legible signal in this toy
2-D actuation setup — Frechet distance is numerically small given the
low-dimensional summary space and minimal training budget, though it still
moves in the same direction as the perturbation (as Part 1's sweep showed for
the richer 258-D perception case). Spec Section 7.2 requires reporting both
metrics together for exactly this reason: they don't always agree in
magnitude, and that disagreement is itself informative rather than something
to average away. `generate_report()` surfaces this automatically as a
rank-disagreement note when there are enough conditions to check it.

## Part 3 — Validation harness: anti-cherry-picking, checked in code

Spec 8.3 requires conditions to be pre-registered *before* any gap score or
ground truth is looked at; `ValidationHarness.run()` raises if the submitted
set doesn't exactly match. Ten tremor-amplitude conditions are registered
below and correlated against a proxy ground truth via Spearman + bootstrap CI.

**Honest caveat**: spec 8.1 requires ground truth computed independently of
the model under test -- for a real V1 run that's raw MediaPipe
confidence/dropout. This notebook has no MediaPipe/webcam access, so it uses
the known perturbation amplitude itself as a stand-in ground truth (it's
independent of the world model by construction, but it is *not* the real
validation spec 8.1 calls for). Treat the correlation number below as a
pipeline check, not a validated result.

In [9]:
amplitudes = np.linspace(0.05, 0.8, 10)
conditions = []
condition_results = []

for amp in amplitudes:
    condition = {"tremor_amplitude": round(float(amp), 3)}
    perturbed = [inject_tremor(r, amplitude=amp, seed=i) for i, r in enumerate(clean_rollouts)]
    result = perception_analyzer.compute_gap(clean_rollouts, perturbed)
    proxy_ground_truth = float(amp)  # see caveat above -- not real MediaPipe confidence
    conditions.append(condition)
    condition_results.append(
        ConditionResult(
            condition=condition,
            gap_score=result.mmd.mmd_squared,
            ground_truth_degradation=proxy_ground_truth,
        )
    )

harness = ValidationHarness(min_conditions=10)
harness.pre_register_conditions(conditions)
validation_report = harness.run(condition_results)

s = validation_report.spearman
print(f"Spearman rho={s.rho:.3f} (95% CI=[{s.ci_low:.3f}, {s.ci_high:.3f}], n_bootstrap={s.n_bootstrap})")
print(f"n_conditions={validation_report.n_conditions}")

Spearman rho=1.000 (95% CI=[1.000, 1.000], n_bootstrap=1000)
n_conditions=10


In [10]:
# Anti-cherry-picking enforcement, demonstrated directly: submitting a subset
# of the pre-registered conditions must raise, not silently succeed.
try:
    harness.run(condition_results[:5])
    raise AssertionError("expected a ValueError -- anti-cherry-picking check did not fire")
except ValueError as e:
    print(f"Correctly rejected a trimmed condition set:\n  {e}")

Correctly rejected a trimmed condition set:
  submitted condition_results do not exactly match the pre-registered set (missing=5, unexpected extra=0) — this is exactly the selective-reporting pattern spec 8.3 prohibits


## Summary

- V1 (perception) gap: computed via `GapAnalyzer`, shown to scale with
  perturbation severity, reported with sample-size confidence flagging
  (spec 7.3), and bundled into an HTML report (spec 9.3).
- V2 (actuation) gap: computed via the **same** `GapAnalyzer` class, only
  `config.modality` changed -- the concrete reusability check.
- Validation harness: pre-registration enforced in code, not just by
  convention -- trimming the condition set after the fact raises.

What this notebook does *not* demonstrate, because it's genuinely blocked in
this environment rather than skipped for convenience: real HaGRID/EgoHands
data (needs MediaPipe + downloaded frames, `ROADMAP.md` Phase 0/1), the real
Ogawa et al. PGM curve (needs digitizing a paywalled figure, Phase 6), and any
GPU-scale training run. See `ROADMAP.md` for current status on all three.